# Multimodal Embedding with Qwen3-VL and OpenVINO

The Qwen3-VL-Embedding model series is built upon the powerful Qwen3-VL foundation model, specifically designed for multimodal embedding tasks. It accepts diverse inputs including text, images, screenshots, and videos, as well as inputs containing a mixture of these modalities. The model generates high-dimensional vectors for broad applications like retrieval, clustering, and classification.

<img src="https://model-demo.oss-cn-hangzhou.aliyuncs.com/Qwen3-VL-Embedding.png" width="500"/>

In this tutorial we consider how to convert and optimize Qwen3-VL Embedding model using OpenVINO.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Select model](#Select-model)
- [Convert model using Optimum Intel](#Convert-model-using-Optimum-Intel)
- [Run OpenVINO model inference with Optimum-intel](#Run-OpenVINO-model-inference-with-Optimum-intel)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/qwen3-vl-embedding/qwen3-vl-embedding.ipynb" />

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [ ]:
import platform

%pip uninstall -q -y optimum optimum-intel optimum-onnx
%pip install "git+https://github.com/openvino-dev-samples/optimum-intel.git@qwen3vl-reranker" "transformers>=4.57.0,<5.8.0" "torch>=2.9" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -qU "openvino>=2025.4" "openvino_tokenizers>=2025.4"

if platform.system() == "Darwin":
    %pip install -q "numpy<2.0.0"

In [1]:
import requests
from pathlib import Path

if not Path("cmd_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py")
    open("cmd_helper.py", "w").write(r.text)

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("qwen3-vl-embedding.ipynb")

## Select model
[back to top ⬆️](#Table-of-contents:)

Qwen3-VL Embedding Model list:

| Model Type | Models | Size | Layers | Sequence Length | Embedding Dimension | MRL Support | Instruction Aware |
|---|---|---|---|---|---|---|---|
| Multimodal Embedding | [Qwen3-VL-Embedding-2B](https://huggingface.co/Qwen/Qwen3-VL-Embedding-2B) | 2B | 28 | 32K | 2048 | Yes | Yes |
| Multimodal Embedding | [Qwen3-VL-Embedding-8B](https://huggingface.co/Qwen/Qwen3-VL-Embedding-8B) | 8B | 36 | 32K | 4096 | Yes | Yes |

In [2]:
import ipywidgets as widgets

model_ids = ["Qwen/Qwen3-VL-Embedding-2B", "Qwen/Qwen3-VL-Embedding-8B"]

model_selector = widgets.Dropdown(
    options=model_ids,
    default=model_ids[0],
    description="Embedding Model:",
)

model_selector

Dropdown(description='Embedding Model:', options=('Qwen/Qwen3-VL-Embedding-2B', 'Qwen/Qwen3-VL-Embedding-8B'),…

## Convert model using Optimum Intel
[back to top ⬆️](#Table-of-contents:)

For convenience, we will use OpenVINO integration with HuggingFace Optimum. [Optimum Intel](https://huggingface.co/docs/optimum/intel/index) is the interface between the Transformers and Diffusers libraries and the different tools and libraries provided by Intel to accelerate end-to-end pipelines on Intel architectures.

Among other use cases, Optimum Intel provides a simple interface to optimize your Transformers and Diffusers models, convert them to the OpenVINO Intermediate Representation (IR) format and run inference using OpenVINO Runtime. `optimum-cli` provides command line interface for model conversion and optimization.

General command format:

```bash
optimum-cli export openvino --model <model_id_or_path> --task <task> <output_dir>
```

where task is task to export the model for. Additionally, you can specify weights compression using `--weight-format` argument with one of following options: `fp32`, `fp16`, `int8` and `int4`.

In [3]:
to_compress = widgets.Checkbox(
    value=False,
    description="Weight compression",
    disabled=False,
)

visible_widgets = [to_compress]

options = widgets.VBox(visible_widgets)

options

The Qwen3-VL-Embedding model can be exported by `feature-extraction` task with Optimum-intel.

In [4]:
from pathlib import Path

model_id = model_selector.value

model_base_dir = Path(model_id.split("/")[-1])
additional_args = {"task": "feature-extraction"}

if to_compress.value:
    model_dir = model_base_dir / "INT8"
    additional_args.update({"weight-format": "int8"})
else:
    model_dir = model_base_dir / "FP16"
    additional_args.update({"weight-format": "fp16"})

In [5]:
from cmd_helper import optimum_cli

if not model_dir.exists():
    optimum_cli(model_id, model_dir, additional_args=additional_args)

**Export command:**

`optimum-cli export openvino --model Qwen/Qwen3-VL-Embedding-2B Qwen3-VL-Embedding-2B\FP16 --task feature-extraction --weight-format fp16`

## Run OpenVINO model inference with Optimum-intel
[back to top ⬆️](#Table-of-contents:)

Select device from dropdown list for running inference using OpenVINO.

In [6]:
from notebook_utils import device_widget

device = device_widget(default="CPU", exclude=["NPU"])
device

Dropdown(description='Device:', options=('CPU', 'GPU', 'AUTO'), value='CPU')

The Qwen3-VL-Embedding model can be loaded by class `OVModelForFeatureExtraction` with Optimum-intel.

In [7]:
from optimum.intel import OVModelForFeatureExtraction

model = OVModelForFeatureExtraction.from_pretrained(model_dir, device=device.value, export=False)

In [ ]:
import torch
import torch.nn.functional as F

from torch import Tensor
from transformers import AutoProcessor


def last_token_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
    left_padding = attention_mask[:, -1].sum() == attention_mask.shape[0]
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]


def format_inputs(processor, inputs, instruction="Represent the user's input."):
    """Format text inputs for the embedding model using chat template."""
    conversations = []
    for inp in inputs:
        content = []
        if isinstance(inp, str):
            content.append({"type": "text", "text": inp})
        elif isinstance(inp, dict):
            if "image" in inp:
                content.append({"type": "image", "image": inp["image"]})
            if "text" in inp:
                content.append({"type": "text", "text": inp["text"]})
        conversation = [
            {"role": "system", "content": [{"type": "text", "text": instruction}]},
            {"role": "user", "content": content},
        ]
        conversations.append(conversation)

    prompts = [
        processor.apply_chat_template(conv, tokenize=False, add_generation_prompt=True)
        for conv in conversations
    ]
    batch = processor(text=prompts, padding=True, return_tensors="pt")
    return batch


processor = AutoProcessor.from_pretrained(model_dir)

# Define text queries and documents
queries = [
    "What is deep learning?",
    "A woman playing with her dog on a beach at sunset.",
]

documents = [
    "Deep learning is a subset of machine learning that uses neural networks with many layers.",
    "A woman shares a joyful moment with her golden retriever on a sun-drenched beach at sunset.",
]

all_texts = queries + documents
batch = format_inputs(processor, all_texts)

outputs = model(**batch)

# Convert to torch tensor if needed (OpenVINO returns numpy arrays)
hidden_state = outputs.last_hidden_state
if not isinstance(hidden_state, torch.Tensor):
    hidden_state = torch.tensor(hidden_state)

embeddings = last_token_pool(hidden_state, batch["attention_mask"])

# Normalize embeddings
embeddings = F.normalize(embeddings, p=2, dim=1)

# Compute similarity scores between queries and documents
scores = embeddings[:2] @ embeddings[2:].T
print("Similarity scores (queries x documents):")
print(scores.tolist())

The tokenizer you are loading from 'Qwen3-VL-Embedding-2B\FP16' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


AttributeError: 'numpy.ndarray' object has no attribute 'norm'

In [ ]:
del model
import gc
gc.collect()